# Fantasy Premier League Analysis

This notebook loads FPL entry histories, builds features, and simulates finish outcomes.

## How to read the charts together

Three charts near the end tell one story, and they should be read as a set.

1. **Change in win probability** (the stacked bars) says *who moved the needle this week*. Each
   week's gains stack above zero and losses below, and they always balance. Moves inside the
   simulation noise are zeroed by a t-test, so a bar is a real shift. A bar with no counterpart
   in earlier weeks is news; a week of small bars both sides is ordinary churn.
2. **Required run rate** says *when the gap stopped being closable*. The dashed line is what a
   manager needs per week from here to catch the leader by GW38. The week it climbs past
   anything they have ever averaged is the week the race was over for them. For the eventual
   winner it sits on their own average.
3. **Form versus season average** says *whether anyone is actually running at the pace needed*.
   The dotted form line is noisy by design. One week above the target means little; a run of
   three or more is a genuine spell. The form-adjusted win probability panel is the same idea
   in probability terms, and the place where the solid and dashed lines split is a manager
   whose level has changed and the season average has not caught up.

Read them in that order: what happened, whether it matters, and whether it is likely to last.

## What last season looked like through them

In the Boys league, George had three real spells (around GW17, GW23 and GW28) where form was
above the required rate and win probability peaked at 12-17%. Each time the target moved up
faster than a season average could follow. Mark's second half was a regime shift: he lost
probability steadily from GW7 to GW16, then gained in almost every week from GW17, with three
consecutive double-digit gains at GW28-30. His form line sat above his own average for the
entire run-in. GW30 is where George's required rate reached 71 a week against an average of 55,
and the race was over. In the THG league the same charts show a tighter finish: Dom's two
decisive weeks were GW23 and GW35, the required rate did not leave anyone's reach until GW31,
and the t-test zeroed only 22% of moves against 36% in the Boys league.

## The model behind the probabilities

Each manager's weekly rate is their running mean, shrunk toward the league average with a prior
worth `PRIOR_GAMEWEEKS` of evidence, and drawn each iteration with its standard error. Weekly
scores are normal with `WEEKLY_POINTS_SD` (16, from last season's within-manager spread), not
Poisson: a Poisson at 55 points has a standard deviation of about 7.5, less than half the real
figure, and it is why the previous version called a 20-point lead at GW3 a certainty. Validated
on both 2025-26 leagues: log-loss on the eventual winner halved, and the old model's eleven
weeks of giving the eventual winner under 1% are gone.

## Setup

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd
import seaborn as sns
from scipy.stats import poisson

# Run from the project folder, so data/ and outputs/ resolve as they do for the CLIs.
PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / "src"))

from fpl_optimiser.leagues import build as league_build
from fpl_optimiser.leagues import export as league_export
from fpl_optimiser.leagues.analysis import (
    add_features,
    add_required_run_rate,
    load_histories,
    simulate_season_finish,
    simulate_win_prob_by_gw,
    win_prob_deltas,
)
from fpl_optimiser.leagues.export import EWM_SPAN, LEAGUES, N_ITER, RECENT_WINDOW, SEED, TOTAL_GWS

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## Parameters

In [ ]:
# Entry ids (reissued every season) and the simulation settings (TOTAL_GWS, N_ITER, SEED,
# EWM_SPAN, RECENT_WINDOW) live in fpl_optimiser/leagues/export.py, so the notebook and the
# league pages can't drift apart. Change them there.
LEAGUE = "boys"  # or 'thg'
TEAM_MAP = LEAGUES[LEAGUE]["entries"]

# One colour per person, in TEAM_MAP order, used by every plot below so a name keeps its colour.
# Ten slots, colour-blind-checked; george is first in both leagues so he is always blue.
COLOURS = [
    "#2a78d6",
    "#eb6834",
    "#1baf7a",
    "#eda100",
    "#e87ba4",
    "#008300",
    "#8e5cd9",
    "#a0522d",
    "#00a3c4",
    "#c0392b",
]
NAME_ORDER = list(TEAM_MAP.values())
PALETTE = dict(zip(NAME_ORDER, COLOURS, strict=False))

OUTPUT_DIR = "data"
REFRESH_DATA = True

## Data Load

In [ ]:
history_df = load_histories(
    TEAM_MAP,
    output_dir=OUTPUT_DIR,
    refresh=REFRESH_DATA,
)
history_df.head()

## Feature Engineering

In [ ]:
feature_df = add_features(history_df, ewma_span=EWM_SPAN)
feature_df.head()

## Exploratory Summary

In [ ]:
feature_df.groupby("name")[["points", "points_on_bench", "total_points"]].describe()

## Exploratory Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)

sns.lineplot(
    data=feature_df,
    x="event",
    y="points",
    hue="name",
    hue_order=NAME_ORDER,
    palette=PALETTE,
    ax=axes[0, 0],
)
axes[0, 0].set_title("Weekly Points")

sns.lineplot(
    data=feature_df,
    x="event",
    y="points_on_bench",
    hue="name",
    hue_order=NAME_ORDER,
    palette=PALETTE,
    ax=axes[0, 1],
)
axes[0, 1].set_title("Bench Points")

sns.lineplot(
    data=feature_df,
    x="event",
    y="total_points",
    hue="name",
    hue_order=NAME_ORDER,
    palette=PALETTE,
    ax=axes[1, 0],
)
axes[1, 0].set_title("Total Points")

sns.lineplot(
    data=feature_df,
    x="event",
    y="ewma_points",
    hue="name",
    hue_order=NAME_ORDER,
    palette=PALETTE,
    ax=axes[1, 1],
)
axes[1, 1].set_title(f"EWMA Points (span={EWM_SPAN})")

for ax in axes.flat:
    ax.legend(loc="best")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
sns.lineplot(
    data=feature_df,
    x="event",
    y="points_from_leader",
    hue="name",
    hue_order=NAME_ORDER,
    palette=PALETTE,
    ax=ax,
)
ax.set_title("Points Behind Leader by Gameweek")
plt.show()

### Running Total Over Last 5 Games

In [ ]:
rolling5_df = feature_df.sort_values(["team_id", "event"]).copy()
rolling5_df["points_last_5"] = rolling5_df.groupby("team_id")["points"].transform(
    lambda s: s.rolling(window=5, min_periods=1).sum()
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(
    data=rolling5_df,
    x="event",
    y="points_last_5",
    hue="name",
    hue_order=NAME_ORDER,
    palette=PALETTE,
    ax=ax,
)
ax.set_title("Running Total Points Over Last 5 Games")
ax.set_xlabel("Gameweek")
ax.set_ylabel("Points in Last 5 Games")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.kdeplot(
    data=feature_df,
    x="points",
    hue="name",
    hue_order=NAME_ORDER,
    palette=PALETTE,
    fill=True,
    common_norm=False,
    alpha=0.25,
    ax=ax,
)
ax.set_title("Distribution of Weekly Points")
plt.show()

In [ ]:
poisson.pmf(0, feature_df["points"].mean(), loc=0)

## Season Finish Simulation

In [ ]:
season_sim_df = simulate_season_finish(
    feature_df,
    TEAM_MAP,
    total_gws=TOTAL_GWS,
    n_iter=N_ITER,
    seed=SEED,
    recent_window=RECENT_WINDOW,
)
season_sim_df.head()

In [ ]:
rank_probs = (
    season_sim_df.groupby(["name", "final_rank"])["iteration"].count().rename("count").reset_index()
)
rank_probs["probability"] = rank_probs.groupby("name")["count"].transform(
    lambda counts: counts / counts.sum()
)
rank_probs = rank_probs.drop(columns="count")
rank_probs

In [ ]:
ax = sns.barplot(
    data=rank_probs,
    x="final_rank",
    y="probability",
    hue="name",
    hue_order=NAME_ORDER,
    palette=PALETTE,
)
ax.set_title("Probability of each position")
ax.set_ylabel("Probability")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend(loc=(1.04, 0))
plt.show()

In [ ]:
recent_rank_probs = (
    season_sim_df.groupby(["name", "final_rank_recent"])["iteration"]
    .count()
    .rename("count")
    .reset_index()
)
recent_rank_probs["probability"] = recent_rank_probs.groupby("name")["count"].transform(
    lambda counts: counts / counts.sum()
)
recent_rank_probs = recent_rank_probs.drop(columns="count")
recent_rank_probs

## Win Probability Over Time

In [ ]:
win_prob_df = simulate_win_prob_by_gw(
    feature_df,
    TEAM_MAP,
    total_gws=TOTAL_GWS,
    n_iter=N_ITER,
    seed=SEED,
)
win_prob_df.head()

In [ ]:
win_prob_columns = [f"{name}_win_prob" for name in NAME_ORDER]
ax = win_prob_df.plot.line(
    x="game_week", y=win_prob_columns, color=[PALETTE[name] for name in NAME_ORDER], figsize=(12, 5)
)
ax.set_title("Probability of Winning the League")
ax.set_ylabel("Probability")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend(NAME_ORDER, loc=(1.04, 0))
plt.show()

In [ ]:
# Week-on-week change in win probability. Each probability is a mean of N_ITER 0/1 draws, so a
# Welch t-test on the two weeks decides whether a change beats the simulation noise; if not, 0.
win_prob_delta_df = win_prob_deltas(win_prob_df, n_iter=N_ITER, alpha=0.05)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(
    data=win_prob_delta_df,
    x="game_week",
    y="delta_significant",
    hue="name",
    hue_order=NAME_ORDER,
    palette=PALETTE,
    ax=ax,
)
ax.axhline(0, color="#888", lw=0.8)
ax.set_title("Change in Win Probability vs Previous Gameweek (noise zeroed, p > 0.05)")
ax.set_xlabel("Gameweek")
ax.set_ylabel("Change in probability")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend(loc=(1.04, 0))
plt.show()

In [ ]:
win_prob_delta_df.pivot(  # noqa: PD010
    index="game_week", columns="name", values="delta_significant"
)[NAME_ORDER].round(3)

## Form-Adjusted Win Probability

The season-mean rate is slow to notice a manager whose level has changed. Where the dashed form line pulls away from the solid one, recent weeks disagree with the season average and the solid line has not caught up yet.

In [ ]:
# Same simulation, but the rate is the EWMA (span EWM_SPAN) instead of the season mean, and it is
# only trusted as EWM_SPAN games of evidence when shrinking toward the league mean.
win_prob_form_df = simulate_win_prob_by_gw(
    feature_df,
    TEAM_MAP,
    total_gws=TOTAL_GWS,
    n_iter=N_ITER,
    seed=SEED,
    rate_column="ewma_points",
    rate_games=EWM_SPAN,
)

n_cols = min(5, len(NAME_ORDER))
n_rows = -(-len(NAME_ORDER) // n_cols)
fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(3.2 * n_cols, 3.4 * n_rows),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
for ax, name in zip(axes.flat, NAME_ORDER, strict=False):
    ax.plot(
        win_prob_df["game_week"],
        win_prob_df[f"{name}_win_prob"],
        color=PALETTE[name],
        lw=2,
        label="season mean",
    )
    ax.plot(
        win_prob_form_df["game_week"],
        win_prob_form_df[f"{name}_win_prob"],
        color=PALETTE[name],
        lw=1.5,
        ls="--",
        label=f"form (EWMA span {EWM_SPAN})",
    )
    ax.set_title(name)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
for ax in axes.flat[len(NAME_ORDER) :]:
    ax.set_visible(False)
axes.flat[0].legend(loc="upper left", fontsize=8)
fig.supxlabel("Gameweek")
fig.supylabel("P(win league)")
fig.suptitle("Win probability: season-mean rate (solid) vs form-weighted rate (dashed)")
plt.show()

## Required Run Rate

Points per week each manager needs from here to catch the leader by GW38: the leader's running mean plus the gap spread over the games left. The week the dashed line climbs past anything a manager has ever averaged is the week the race was really over.

In [ ]:
run_rate_df = add_required_run_rate(feature_df, total_gws=TOTAL_GWS)
league_mean = feature_df["points"].mean()

n_cols = min(5, len(NAME_ORDER))
n_rows = -(-len(NAME_ORDER) // n_cols)
fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(3.2 * n_cols, 3.4 * n_rows),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)
for ax, name in zip(axes.flat, NAME_ORDER, strict=False):
    person = run_rate_df.loc[run_rate_df["name"] == name]
    ax.plot(
        person["event"],
        person["running_mean_points"],
        color=PALETTE[name],
        lw=2,
        label="own season average",
    )
    ax.plot(
        person["event"],
        person["ewma_points"],
        color=PALETTE[name],
        lw=1.5,
        ls=":",
        label=f"own form (EWMA span {EWM_SPAN})",
    )
    ax.plot(
        person["event"],
        person["required_rate"],
        color="#666",
        lw=1.5,
        ls="--",
        label="needed to catch leader",
    )
    ax.set_title(name)
    ax.set_ylim(
        0, 2 * league_mean
    )  # the required rate explodes in the run-in; off the top means "gone"
for ax in axes.flat[len(NAME_ORDER) :]:
    ax.set_visible(False)
axes.flat[0].legend(loc="upper left", fontsize=8)
fig.supxlabel("Gameweek")
fig.supylabel("Points per week")
fig.suptitle("Required run rate to catch the leader vs own season average and form")
plt.show()

In [ ]:
win_prob_melt_df = pd.melt(win_prob_df, id_vars="game_week").rename(
    columns={"value": "win_probability"}
)
win_prob_melt_df["name"] = win_prob_melt_df.variable.str.split("_", n=1, expand=True)[0]

win_prob_melt_df = win_prob_melt_df.merge(
    feature_df,
    how="left",
    right_on=["event", "name"],
    left_on=["game_week", "name"],
)
win_prob_melt_df["pflpgw"] = win_prob_melt_df["points_from_leader"] / (
    38 - win_prob_melt_df["game_week"]
)

In [ ]:
win_prob_melt_df.loc[win_prob_melt_df["game_week"] >= 10].corr(numeric_only=True)

In [ ]:
win_prob_melt_df.loc[win_prob_melt_df["game_week"] >= 10].plot.scatter(
    x="win_probability", y="running_mean_points"
)

In [ ]:
win_prob_melt_df.loc[win_prob_melt_df["game_week"] >= 10].plot.scatter(
    x="win_probability", y="points_from_leader"
)

In [ ]:
win_prob_melt_df.loc[win_prob_melt_df["game_week"] >= 10].plot.scatter(
    x="win_probability", y="pflpgw"
)

## League pages

Rebuilds both league pages (`league_pages/out/boys_fpl.html` and `thg_fpl.html`) with the same simulation, plus the gameweek detail and the season transfer tally. Add the new week's `league_pages/commentary/<league>/gwNN.md` first. Publishing is a separate step: ask Claude to republish the two files to their existing links.

In [ ]:
league_export.run(cached=not REFRESH_DATA)
league_build.main()

league_payload = json.loads((league_export.OUT_DIR / f"{LEAGUE}.json").read_text())["data"]

In [ ]:
# Season transfer tally: each signing scored from arrival until sold again (or now) against the
# player sold, over the same weeks, less hits. Best/worst use normal transfers only.
def describe_move(move: dict | None) -> str:
    if move is None:
        return ""
    return f"{move['out']} -> {move['in']} ({move['gain']:+d}, GW{move['gw']}, {move['weeks']} wk)"


season_transfers_df = (
    pd.DataFrame(
        [
            {
                "name": name,
                "moves": r["transfers"],
                "hits": r["hits"],
                "transfers_net": r["gain_regular"],
                "chip_squads_net": r["gain_chips"],
                "total": r["gain"],
                "best_move": describe_move(r["best"]),
                "worst_move": describe_move(r["worst"]),
                "chips": ", ".join(f"{c['chip']} GW{c['gw']}" for c in r["chips"]),
            }
            for name, r in league_payload["season_transfers"].items()
        ]
    )
    .sort_values("total", ascending=False)
    .reset_index(drop=True)
)
season_transfers_df

In [ ]:
# Latest gameweek, one row per manager: star = best XI score incl. captaincy; transfers_net =
# points in minus points out that week, after any hit; benched = beat a same-role starter by 2+.
latest_gw = max(league_payload["gameweeks"], key=int)
latest_gw_df = (
    pd.DataFrame(
        [
            {
                "name": name,
                "points": r["points"],
                "hit": r["hit"],
                "star": f"{r['star']['player']} {r['star']['points']}",
                "captain": f"{r['captain']['player']} {r['captain']['points']}"
                + (" (vice)" if r["captain"]["vice_stood_in"] else ""),
                "transfers": "; ".join(
                    f"{t['out']} {t['out_points']} -> {t['in']} {t['in_points']}"
                    for t in r["transfers"]
                ),
                "transfers_net": r["transfer_net"],
                "chip": r["chip"] or "",
                "benched": ", ".join(f"{m['player']} {m['points']}" for m in r["missed"]),
            }
            for name, r in league_payload["gameweeks"][latest_gw].items()
        ]
    )
    .sort_values("points", ascending=False)
    .reset_index(drop=True)
)
print(f"GW{latest_gw}")
latest_gw_df